# MS3 — Step 4: AlphaFold3 Core Predictions (Full 14-Allele Panel)

Generates AF3 inputs for every HLA-peptide pair produced by Step 2
(`data/step2_iedb_epitopes.csv`, ~30-40 pairs across the 14-allele panel),
tracks manual submission against the AF3 free-tier rate limit (~10-20 jobs/day),
and ingests results as they come in.

**This is the largest and slowest step in the MS3 pipeline** — with 30-40 pairs
against a ~10-20/day free-tier limit, expect submission to span roughly 2-5 days
of real time, done in daily batches rather than all at once.

**Compute stack:** AlphaFold3 server (free, manual web submission, no batch API).

**Workflow:**
1. Load Step 2's epitope list, generate AF3 JSON inputs for all pairs (Part A)
2. Get today's submission batch respecting the daily rate limit (Part B)
3. **[MANUAL, REPEATED DAILY]** Submit today's batch, download results
4. Ingest whatever results have arrived, update the manifest (Part C)
5. Repeat B-C until all pairs are done
6. Export final structure manifest for Step 5 / Step 6 (Part D)

## Setup

In [1]:
!pip install -q pandas requests 2>/dev/null || pip install -q --break-system-packages pandas requests

import json, csv
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional
from datetime import date

import pandas as pd

BASE_DIR = Path("/content/ms3_step4")
FASTA_DIR = BASE_DIR / "af3_inputs"
RESULTS_DIR = BASE_DIR / "af3_results"       # <- drop downloaded AF3 result folders here
MANIFEST_PATH = BASE_DIR / "step4_manifest.csv"

for d in (FASTA_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Upload data/step2_iedb_epitopes.csv here if not already present
STEP2_CSV = BASE_DIR / "step2_iedb_epitopes.csv"
print(f"Working directory: {BASE_DIR}")
print(f"Expected Step 2 input: {STEP2_CSV}")
print(f"(Upload it below if it isn't already there)")

Working directory: /content/ms3_step4
Expected Step 2 input: /content/ms3_step4/step2_iedb_epitopes.csv
(Upload it below if it isn't already there)


In [2]:
# Upload Step 2's output CSV if it isn't already sitting in BASE_DIR
try:
    from google.colab import files
    if not STEP2_CSV.exists():
        print("Upload data/step2_iedb_epitopes.csv:")
        uploaded = files.upload()
        for fname, content in uploaded.items():
            (BASE_DIR / fname).write_bytes(content)
        # Rename to expected filename if needed
        candidates = list(BASE_DIR.glob("*iedb_epitopes*.csv")) + list(BASE_DIR.glob("*step2*.csv"))
        if candidates and not STEP2_CSV.exists():
            candidates[0].rename(STEP2_CSV)
except ImportError:
    pass

if STEP2_CSV.exists():
    print(f"Found: {STEP2_CSV}")
else:
    print(f"[WARNING] {STEP2_CSV} not found — place it there before continuing.")

Upload data/step2_iedb_epitopes.csv:


Saving step2_iedb_epitopes.csv to step2_iedb_epitopes.csv
Found: /content/ms3_step4/step2_iedb_epitopes.csv


## Part A — Load Epitope Pairs and Generate AF3 Inputs

Reads Step 2's output, normalises allele names, and builds one AF3 JSON job
per pair using the full 14-allele sequence dictionary.

In [3]:
HLA_ALPHA_SEQS: dict[str, str] = {
    # HLA-A
    "HLA-A*01:01": "GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGYYNQSEDGSHTIQIMYGCDVGPDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRWE",
    "HLA-A*02:01": "GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQKMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGYYNQSEDGSHTIQIMYGCDVGPDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRWE",
    "HLA-A*03:01": "GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGYYNQSEDGSHTIQIMYGCDVGPDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRWE",
    "HLA-A*24:02": "GSHSMRYFHTSVSRPGRGEPRFISVGYVDDTQFVRFDSDAASPREEPRAPWIEQEGPEYWDRETQICKAHSQTHRVDLGTLRGYYNQSEDDSHTIQIMYGCDVGPDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQITKRKWEAAHVAEQQRAYLEGTCVEWLRRYLENGKETLQRTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDWTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWE",
    # HLA-B
    "HLA-B*07:02": "GSHSMRYFYTSVSRPGRGEPRFISVGYVDDTQFVRFDSDAASPRGEPRAPWIEQEGPEYWDRETQKYKAHSQTHRVDLGTLRGYYNQSELDGSHTIQIMYGCDVGPDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQITKRKWEAAHVAEQQRAYLEGTCVEWLRRYLENGKETLQRTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDWTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWE",
    "HLA-B*08:01": "GSHSMRYFFTSVSRPGRGEPRFISVGYVDDTQFVRFDSDAASPREEPRAPWIEQEGPEYWDRETQKYKAHSQTHRVDLGTLRGYYNQSEAGSHTIQIMYGCDVGPDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADTAAQITKRKWEAAHVAEQQRAYLEGTCVEWLRRYLENGKETLQRTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDWTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWE",
    "HLA-B*15:01": "GSHSMRYFYTAVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPREEPRAPWIEQEGPEYWDRETQKYKAHSQTHRVDLGTLRGYYNQSEAGSHTIQIMYGCDVGSDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQITKRKWEAAHVAEQQRAYLEGTCVEWLRRYLENGKETLQRTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDWTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWE",
    "HLA-B*35:01": "GSHSMRYFFTAVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPREEPRAPWIEQEGPEYWDRETQKYKAHSQTHRVDLGTLRGYYNQSEAGSHTIQIMYGCDVGPDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQITKRKWEAAHVAEQQRAYLEGTCVEWLRRYLENGKETLQRTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDWTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWE",
    "HLA-B*40:01": "GSHSMRYFFTAVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPREEPRAPWIEQEGPEYWDRETQKYKAHSQTHRVDLGTLRGYYNQSEAGSHTIQIMYGCDVGPDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQITKRKWEAAHVAEQQRAYLEGTCVEWLRRYLENGKETLQRTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDWTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWE",
    "HLA-B*44:02": "GSHSMRYFYTAVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPRGEPRAPWIEQEGPEYWDRETQKYKAHSQTHRVDLGTLRGYYNQSEAGSHTIQIMYGCDVGSDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQITKRKWEAAHVAEQQRAYLEGTCVEWLRRYLENGKETLQRTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDWTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWE",
    # HLA-DRB1 beta chains (DRA alpha chain is invariant)
    "HLA-DRB1*03:01": "GDTRPRFLWQLKFECHFFNGTERVRFLHIYNQEEYARFDSDVGEFRAVTELGRPDAEYWNSQKDLLEQRRAAVDTYCRHNYGVGESFTVQRRVEPKVTVYPSKTQPLQHHNLLVCSVSDFYPGSIQVRWFRNGQEEKAGVVSTGLIQNGDWTFQTLVMLETVPRSGEVYTCQVEHPSVTSPLTVEWRARSESAQSKMLSGVGGFVLGLLFLGAGLLFYLRQQKGHSGLQPTGFLS",
    "HLA-DRB1*04:01": "GDTRPRFLWQLKFECHFFNGTERVRFLHIYNQEEYVRFDSDVGEYRAVTELGRPDAEYWNSQKDLLEQRRAAVDTYCRHNYGVGESFTVQRRVEPKVTVYPSKTQPLQHHNLLVCSVSDFYPGSIQVRWFRNGQEEKAGVVSTGLIQNGDWTFQTLVMLETVPRSGEVYTCQVEHPSVTSPLTVEWRARSESAQSKMLSGVGGFVLGLLFLGAGLLFYLRQQKGHSGLQPTGFLS",
    "HLA-DRB1*07:01": "GDTRPRFLWQLKFECHFFNGTERVRFLHIYNQEEYARFDSDVGEYRAVTELGRPDAEYWNSQKDLLEQRRAAVDTYCRHNYGVGESFTVQRRVEPKVTVYPSKTQPLQHHNLLVCSVSDFYPGSIQVRWFRNGQEEKAGVVSTGLIQNGDWTFQTLVMLETVPRSGEVYTCQVEHPSVTSPLTVEWRARSESAQSKMLSGVGGFVLGLLFLGAGLLFYLRQQKGHSGLQPTGFLS",
    "HLA-DRB1*11:01": "GDTRPRFLWQLKFECHFFNGTERVRFLHIYNQEEYARFDSDVGEFRAVTELGRPDAEYWNSQKDLLEQRRAAVDTYCRHNYGVGESFTVQRRVEPKVTVYPSKTQPLQHHNLLVCSVSDFYPGSIQVRWFRNGQEEKAGVVSTGLIQNGDWTFQTLVMLETVPRSGEVYTCQVEHPSVTSPLTVEWRARSESAQSKMLSGVGGFVLGLLFLGAGLLFYLRQQKGHSGLQPTGFLS",
    "HLA-DRB1*13:01": "GDTRPRFLWQLKFECHFFNGTERVRFLHIYNQEEYARFDSDVGEYRAVTELGRPDAEYWNSQKDLLEQRRAAVDTYCRHNYGVGESFTVQRRVEPKVTVYPSKTQPLQHHNLLVCSVSDFYPGSIQVRWFRNGQEEKAGVVSTGLIQNGDWTFQTLVMLETVPRSGEVYTCQVEHPSVTSPLTVEWRARSESAQSKMLSGVGGFVLGLLFLGAGLLFYLRQQKGHSGLQPTGFLS",
    "HLA-DRB1*15:01": "GDTRPRFLWQLKFECHFFNGTERVRFLHIYNQEEYARFDSDVGEYRAVTELGRPDAEYWNSQKDLLEQRRAAVDTYCRHNYGVGESFTVQRRVEPKVTVYPSKTQPLQHHNLLVCSVSDFYPGSIQVRWFRNGQEEKAGVVSTGLIQNGDWTFQTLVMLETVPRSGEVYTCQVEHPSVTSPLTVEWRARSESAQSKMLSGVGGFVLGLLFLGAGLLFYLRQQKGHSGLQPTGFLS",
}

DRA_SEQ = "MAISGVPVLGFFIIAVLMSAQESWAIKEEHVIIQAEFYLNPDQSGEFMFDFDGDEIFHVDMAKKETVWRLEEFGRFASFEAQGALANIAVDKANLEIMTKRSNYTPITNVPPEVTVLTNSPVELREPNVLICFIDKFTPPVVNVTWLRNGKPVTTGVSETVFLPREDHLFRKFHYLPFLPSTEDVYDCRVEHWGLDEPLLKHWEFDAPSPLPETTENVVCALGLTVGLVGIIIGTIFIKQKGSSS"
B2M_SEQ  = "MSRSVALAVLALLSLSGLEAIQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDWSFYLLYYTEFTPTEKDEYACRVNHVTLSQPKIVKWDRDM"

def is_class_ii(locus_or_allele: str) -> bool:
    return "DRB1" in locus_or_allele

print(f"Sequence dictionary loaded: {len(HLA_ALPHA_SEQS)} alleles")

Sequence dictionary loaded: 16 alleles


In [4]:
@dataclass
class EpitopePair:
    pair_id: str
    allele: str
    locus: str
    peptide: str
    epitope_id: str
    antigen: str
    donors: int
    ic50: Optional[float]

def load_epitope_pairs(csv_path: Path) -> list:
    pairs = []
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            try:
                ic50 = float(row["mhc_ic50_nm"]) if row.get("mhc_ic50_nm") else None
            except (ValueError, TypeError):
                ic50 = None
            try:
                donors = int(row.get("positive_subjects") or 0)
            except (ValueError, TypeError):
                donors = 0

            raw_allele = row["allele"]
            fields = raw_allele.replace("HLA-", "").split("*")
            locus_prefix = fields[0]
            digits = fields[1].split(":") if len(fields) > 1 else ["", ""]
            allele_2f = f"HLA-{locus_prefix}*{digits[0]}:{digits[1][:2]}" if len(digits) > 1 else raw_allele

            pairs.append(EpitopePair(
                pair_id=f"P{i + 1:03d}", allele=allele_2f, locus=row["locus"],
                peptide=row["linear_sequence"].strip().upper(),
                epitope_id=row.get("epitope_id", ""), antigen=row.get("antigen_name", ""),
                donors=donors, ic50=ic50,
            ))
    return pairs

if STEP2_CSV.exists():
    epitope_pairs = load_epitope_pairs(STEP2_CSV)
    print(f"Loaded {len(epitope_pairs)} epitope pairs")
    unknown = {p.allele for p in epitope_pairs if p.allele not in HLA_ALPHA_SEQS}
    if unknown:
        print(f"[WARNING] {len(unknown)} allele(s) have no stored sequence and will be skipped: {sorted(unknown)}")
else:
    epitope_pairs = []
    print("[BLOCKED] Cannot proceed without Step 2 CSV. Upload it in the cell above.")

Loaded 932 epitope pairs


In [5]:
def build_af3_job(pair) -> Optional[dict]:
    alpha_seq = HLA_ALPHA_SEQS.get(pair.allele)
    if alpha_seq is None:
        return None

    if is_class_ii(pair.locus):
        sequences = [
            {"proteinChain": {"sequence": DRA_SEQ, "count": 1}},
            {"proteinChain": {"sequence": alpha_seq, "count": 1}},
            {"proteinChain": {"sequence": pair.peptide, "count": 1}},
        ]
    else:
        sequences = [
            {"proteinChain": {"sequence": alpha_seq, "count": 1}},
            {"proteinChain": {"sequence": B2M_SEQ, "count": 1}},
            {"proteinChain": {"sequence": pair.peptide, "count": 1}},
        ]

    return {
        "name": f"MS3_step4_{pair.pair_id}",
        "modelSeeds": [1],
        "sequences": sequences,
        "dialect": "alphafoldserver",
        "version": 1,
    }

# Generate all AF3 input files and the master manifest
manifest_rows = []
for pair in epitope_pairs:
    job = build_af3_job(pair)
    if job is None:
        manifest_rows.append({**asdict(pair), "af3_input_file": None,
                               "expected_result_folder": None,
                               "submitted_date": None, "downloaded": False,
                               "status": "skipped_no_sequence"})
        continue

    job_path = FASTA_DIR / f"{pair.pair_id}_af3input.json"
    job_path.write_text(json.dumps([job], indent=2))
    expected_dir = RESULTS_DIR / pair.pair_id

    manifest_rows.append({**asdict(pair), "af3_input_file": str(job_path),
                           "expected_result_folder": str(expected_dir),
                           "submitted_date": None, "downloaded": False,
                           "status": "pending"})

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(MANIFEST_PATH, index=False)
n_ready = (manifest_df["status"] == "pending").sum()
n_skipped = (manifest_df["status"] == "skipped_no_sequence").sum()
print(f"Generated {n_ready} AF3 input files ({n_skipped} skipped, no stored sequence)")
print(f"Manifest saved: {MANIFEST_PATH}")
manifest_df.head(10)

Generated 932 AF3 input files (0 skipped, no stored sequence)
Manifest saved: /content/ms3_step4/step4_manifest.csv


,pair_id,allele,locus,peptide,epitope_id,antigen,donors,ic50,af3_input_file,expected_result_folder,submitted_date,downloaded,status
0,P001,HLA-A*01:01,HLA-A,TTDPSFLGRY,,orf1ab polyprotein [Severe acute respiratory s...,0,None,/content/ms3_step4/af3_inputs/P001_af3input.json,/content/ms3_step4/af3_results/P001,None,False,pending
1,P002,HLA-A*01:01,HLA-A,TTDPSFLGRY,,orf1ab polyprotein [Severe acute respiratory s...,0,None,/content/ms3_step4/af3_inputs/P002_af3input.json,/content/ms3_step4/af3_results/P002,None,False,pending
2,P003,HLA-A*01:01,HLA-A,TTDPSFLGRY,,orf1ab polyprotein [Severe acute respiratory s...,0,None,/content/ms3_step4/af3_inputs/P003_af3input.json,/content/ms3_step4/af3_results/P003,None,False,pending
3,P004,HLA-A*01:01,HLA-A,TTDPSFLGRY,,orf1ab polyprotein [Severe acute respiratory s...,0,None,/content/ms3_step4/af3_inputs/P004_af3input.json,/content/ms3_step4/af3_results/P004,None,False,pending
4,P005,HLA-A*01:01,HLA-A,TTDPSFLGRY,,orf1ab polyprotein [Severe acute respiratory s...,0,None,/content/ms3_step4/af3_inputs/P005_af3input.json,/content/ms3_step4/af3_results/P005,None,False,pending
5,P006,HLA-A*01:01,HLA-A,PTDNYITTY,,orf1ab polyprotein [Severe acute respiratory s...,0,None,/content/ms3_step4/af3_inputs/P006_af3input.json,/content/ms3_step4/af3_results/P006,None,False,pending
6,P007,HLA-A*01:01,HLA-A,ATSRTLSYY,,membrane glycoprotein [Severe acute respirator...,0,None,/content/ms3_step4/af3_inputs/P007_af3input.json,/content/ms3_step4/af3_results/P007,None,False,pending
7,P008,HLA-A*01:01,HLA-A,CTDDNALAYY,,orf1ab polyprotein [Severe acute respiratory s...,0,None,/content/ms3_step4/af3_inputs/P008_af3input.json,/content/ms3_step4/af3_results/P008,None,False,pending
8,P009,HLA-A*01:01,HLA-A,NTCDGTTFTY,,orf1ab polyprotein [Severe acute respiratory s...,0,None,/content/ms3_step4/af3_inputs/P009_af3input.json,/content/ms3_step4/af3_results/P009,None,False,pending
9,P010,HLA-A*01:01,HLA-A,DTDFVNEFY,,orf1ab polyprotein [Severe acute respiratory s...,0,None,/content/ms3_step4/af3_inputs/P010_af3input.json,/content/ms3_step4/af3_results/P010,None,False,pending


In [6]:
import zipfile
from google.colab import files

zip_path = BASE_DIR / "af3_inputs_all.zip"

# Create a zip file containing all JSON inputs
with zipfile.ZipFile(zip_path, 'w') as zipf:
    json_files = list(FASTA_DIR.glob("*.json"))
    for file in json_files:
        zipf.write(file, arcname=file.name)

print(f"Created zip archive with {len(json_files)} files: {zip_path}")

# Trigger browser download
files.download(str(zip_path))

Created zip archive with 932 files: /content/ms3_step4/af3_inputs_all.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Part B — Daily Submission Batch

Given the free-tier rate limit (~10-20 jobs/day), work through the panel in
daily batches rather than submitting everything at once. Run this cell each
day to get the next batch of pairs still pending submission.

In [7]:
DAILY_BATCH_SIZE = 15  # stay comfortably under the ~10-20/day free-tier limit

def get_todays_batch(manifest_df: pd.DataFrame, batch_size: int = DAILY_BATCH_SIZE) -> pd.DataFrame:
    pending = manifest_df[manifest_df["status"] == "pending"]
    return pending.head(batch_size)

todays_batch = get_todays_batch(manifest_df)
print(f"Today's batch: {len(todays_batch)} pairs")
print(f"({(manifest_df['status'] == 'pending').sum() - len(todays_batch)} will remain for subsequent days)\n")

for _, row in todays_batch.iterrows():
    print(f"  {row['pair_id']}: {row['allele']} / {row['peptide']}  ->  {row['af3_input_file']}")

Today's batch: 15 pairs
(917 will remain for subsequent days)

  P001: HLA-A*01:01 / TTDPSFLGRY  ->  /content/ms3_step4/af3_inputs/P001_af3input.json
  P002: HLA-A*01:01 / TTDPSFLGRY  ->  /content/ms3_step4/af3_inputs/P002_af3input.json
  P003: HLA-A*01:01 / TTDPSFLGRY  ->  /content/ms3_step4/af3_inputs/P003_af3input.json
  P004: HLA-A*01:01 / TTDPSFLGRY  ->  /content/ms3_step4/af3_inputs/P004_af3input.json
  P005: HLA-A*01:01 / TTDPSFLGRY  ->  /content/ms3_step4/af3_inputs/P005_af3input.json
  P006: HLA-A*01:01 / PTDNYITTY  ->  /content/ms3_step4/af3_inputs/P006_af3input.json
  P007: HLA-A*01:01 / ATSRTLSYY  ->  /content/ms3_step4/af3_inputs/P007_af3input.json
  P008: HLA-A*01:01 / CTDDNALAYY  ->  /content/ms3_step4/af3_inputs/P008_af3input.json
  P009: HLA-A*01:01 / NTCDGTTFTY  ->  /content/ms3_step4/af3_inputs/P009_af3input.json
  P010: HLA-A*01:01 / DTDFVNEFY  ->  /content/ms3_step4/af3_inputs/P010_af3input.json
  P011: HLA-A*01:01 / GTDLEGNFY  ->  /content/ms3_step4/af3_inputs/P01

In [8]:
import pandas as pd
manifest = pd.read_csv("/content/ms3_step4/step4_manifest.csv")
unique_pairs = manifest.drop_duplicates(subset=["allele", "peptide"])
print(f"Total rows: {len(manifest)}")
print(f"Unique (allele, peptide) pairs: {len(unique_pairs)}")
print(f"\nDuplication factor: {len(manifest) / len(unique_pairs):.1f}x")
print(f"\nTop duplicated pairs:")
print(manifest.groupby(["allele", "peptide"]).size().sort_values(ascending=False).head(10))

Total rows: 932
Unique (allele, peptide) pairs: 584

Duplication factor: 1.6x

Top duplicated pairs:
allele       peptide   
HLA-B*07:02  SPRWYFYYL     28
HLA-A*01:01  TTDPSFLGRY    19
HLA-A*24:02  RYRIGNYKL     14
HLA-B*40:01  MEVTPSGTWL    14
HLA-A*02:01  LLLDRLNQL     13
             KLWAQCVQL     11
HLA-A*01:01  CTDDNALAYY     9
HLA-A*24:02  VYIGDPAQL      9
HLA-B*07:02  KPRQKRTAT      8
HLA-A*03:01  KTFPPTEPK      8
dtype: int64


In [9]:
import pandas as pd
manifest = pd.read_csv("/content/ms3_step4/step4_manifest.csv")
unique_pairs = manifest.drop_duplicates(subset=["allele", "peptide"]).copy()

print("Donor count distribution across unique pairs:")
print(unique_pairs["donors"].describe())
print(f"\nRows with donors == 0: {(unique_pairs['donors'] == 0).sum()} / {len(unique_pairs)}")
print(f"\nRows with a non-null ic50: {unique_pairs['ic50'].notna().sum()} / {len(unique_pairs)}")

Donor count distribution across unique pairs:
count    584.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: donors, dtype: float64

Rows with donors == 0: 584 / 584

Rows with a non-null ic50: 0 / 584


In [10]:
import pandas as pd

manifest = pd.read_csv("/content/ms3_step4/step4_manifest.csv")

# Record count per unique (allele, peptide) pair is the only genuinely
# populated signal in this manifest -- donors and ic50 are uniformly
# null/zero across all 584 pairs, confirmed directly, so they cannot be
# used for ranking as originally designed. This is a fallback ranking
# using record count as an evidence-strength proxy, not the donor/IC50
# ranking Step 2 was meant to produce. Step 2's IEDB field extraction
# needs a real fix before this pipeline can honestly claim the originally
# intended selection criteria.
pair_counts = (
    manifest.groupby(["allele", "peptide"])
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

TOP_N = 35  # within the original 30-40 pair design target

top_pairs = pair_counts.head(TOP_N)
print(f"Top {TOP_N} pairs by record count (fallback ranking):")
print(top_pairs.to_string(index=False))

# Also check allele diversity in the shortlist -- a ranking by raw record
# count alone could accidentally concentrate on 2-3 alleles with lots of
# IEDB records and lose coverage of the other alleles in the panel
print(f"\nAlleles represented in top {TOP_N}: {sorted(top_pairs['allele'].unique())}")
print(f"Alleles in full panel: {sorted(manifest['allele'].unique())}")

# Build the final shortlisted manifest: one row per selected pair, keeping
# the first matching original row's metadata (antigen, epitope_id, etc.)
shortlist = manifest.merge(
    top_pairs[["allele", "peptide"]], on=["allele", "peptide"], how="inner"
).drop_duplicates(subset=["allele", "peptide"], keep="first")

shortlist_path = "/content/ms3_step4/step4_manifest_SHORTLIST.csv"
shortlist.to_csv(shortlist_path, index=False)
print(f"\nShortlisted manifest saved: {shortlist_path} ({len(shortlist)} pairs)")

Top 35 pairs by record count (fallback ranking):
     allele     peptide  record_count
HLA-B*07:02   SPRWYFYYL            28
HLA-A*01:01  TTDPSFLGRY            19
HLA-A*24:02   RYRIGNYKL            14
HLA-B*40:01  MEVTPSGTWL            14
HLA-A*02:01   LLLDRLNQL            13
HLA-A*02:01   KLWAQCVQL            11
HLA-A*01:01  CTDDNALAYY             9
HLA-A*24:02   VYIGDPAQL             9
HLA-B*07:02   KPRQKRTAT             8
HLA-A*03:01   KTFPPTEPK             8
HLA-A*02:01   FLAFVVFLL             7
HLA-A*02:01  LLLLDRLNQL             7
HLA-A*02:01   LALLLLDRL             7
HLA-B*07:02   IPRRNVATL             7
HLA-A*02:01   GMSRIGMEV             7
HLA-A*02:01   FLLVTLAIL             6
HLA-B*07:02   SPRWYFYYI             6
HLA-A*02:01   FLLNKEMYL             6
HLA-B*07:02   FPRGQGVPI             6
HLA-A*01:01   PTDNYITTY             6
HLA-A*02:01   ALWEIQQVV             5
HLA-A*01:01   DTDFVNEFY             5
HLA-A*02:01   YLATALLTL             5
HLA-A*24:02   FYAYLRKHF             4
H

In [11]:
import pandas as pd

manifest = pd.read_csv("/content/ms3_step4/step4_manifest.csv")

pair_counts = (
    manifest.groupby(["allele", "peptide"])
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

FULL_PANEL = [
    "HLA-A*01:01", "HLA-A*02:01", "HLA-A*03:01", "HLA-A*24:02",
    "HLA-B*07:02", "HLA-B*08:01", "HLA-B*15:01", "HLA-B*35:01",
    "HLA-B*40:01", "HLA-B*44:02",
    "HLA-DRB1*03:01", "HLA-DRB1*04:01", "HLA-DRB1*07:01",
    "HLA-DRB1*11:01", "HLA-DRB1*13:01", "HLA-DRB1*15:01",
]

MIN_PER_ALLELE = 2   # guarantee at least this many pairs per panel allele, if available
TARGET_TOTAL = 35    # overall shortlist size target (30-40 per original design)

# Step 1: for every allele in the full panel, take its top MIN_PER_ALLELE
# pairs by record count -- this guarantees no allele, including the DRB1
# class II arm, is silently dropped the way pure record-count ranking did
guaranteed_rows = []
missing_alleles = []
for allele in FULL_PANEL:
    allele_pairs = pair_counts[pair_counts["allele"] == allele].head(MIN_PER_ALLELE)
    if allele_pairs.empty:
        missing_alleles.append(allele)
    guaranteed_rows.append(allele_pairs)

guaranteed = pd.concat(guaranteed_rows, ignore_index=True) if guaranteed_rows else pd.DataFrame()

if missing_alleles:
    print(f"[WARNING] These alleles have ZERO qualifying pairs in the manifest at all "
          f"(not just low-ranked -- genuinely absent from Step 2's output): {missing_alleles}")
    print("This needs investigating in Step 2 before proceeding -- it means IEDB either has "
          "no qualifying epitopes for these alleles under the current filters, or the query "
          "silently failed for them.")

# Step 2: fill remaining slots up to TARGET_TOTAL with the next-best pairs
# by record count, excluding what's already guaranteed
remaining_slots = max(0, TARGET_TOTAL - len(guaranteed))
already_selected = set(zip(guaranteed["allele"], guaranteed["peptide"]))
fill_candidates = pair_counts[
    ~pair_counts.apply(lambda r: (r["allele"], r["peptide"]) in already_selected, axis=1)
]
fill_rows = fill_candidates.head(remaining_slots)

final_shortlist_pairs = pd.concat([guaranteed, fill_rows], ignore_index=True)

print(f"\nFinal shortlist: {len(final_shortlist_pairs)} pairs")
print(f"Alleles covered: {sorted(final_shortlist_pairs['allele'].unique())}")
print(f"\nPer-allele counts:")
print(final_shortlist_pairs.groupby("allele").size().reindex(FULL_PANEL, fill_value=0))

shortlist = manifest.merge(
    final_shortlist_pairs[["allele", "peptide"]], on=["allele", "peptide"], how="inner"
).drop_duplicates(subset=["allele", "peptide"], keep="first")

shortlist_path = "/content/ms3_step4/step4_manifest_SHORTLIST.csv"
shortlist.to_csv(shortlist_path, index=False)
print(f"\nCoverage-aware shortlist saved: {shortlist_path} ({len(shortlist)} pairs)")


Final shortlist: 35 pairs
Alleles covered: ['HLA-A*01:01', 'HLA-A*02:01', 'HLA-A*03:01', 'HLA-A*24:02', 'HLA-B*07:02', 'HLA-B*08:01', 'HLA-B*15:01', 'HLA-B*35:01', 'HLA-B*40:01', 'HLA-B*44:02', 'HLA-DRB1*03:01', 'HLA-DRB1*04:01', 'HLA-DRB1*07:01', 'HLA-DRB1*11:01', 'HLA-DRB1*13:01', 'HLA-DRB1*15:01']

Per-allele counts:
allele
HLA-A*01:01       2
HLA-A*02:01       5
HLA-A*03:01       2
HLA-A*24:02       2
HLA-B*07:02       2
HLA-B*08:01       2
HLA-B*15:01       2
HLA-B*35:01       2
HLA-B*40:01       2
HLA-B*44:02       2
HLA-DRB1*03:01    2
HLA-DRB1*04:01    2
HLA-DRB1*07:01    2
HLA-DRB1*11:01    2
HLA-DRB1*13:01    2
HLA-DRB1*15:01    2
dtype: int64

Coverage-aware shortlist saved: /content/ms3_step4/step4_manifest_SHORTLIST.csv (35 pairs)


In [12]:
import json
import pandas as pd
from pathlib import Path

shortlist = pd.read_csv("/content/ms3_step4/step4_manifest_SHORTLIST.csv")

combined_jobs = []
missing_files = []

for _, row in shortlist.iterrows():
    json_path = Path(row["af3_input_file"])
    if not json_path.exists():
        missing_files.append(row["pair_id"])
        continue
    # Each individual file on disk is already [job] (a one-item list, per
    # our existing generation code) -- unwrap it and add the single job
    # dict to the combined list
    job_list = json.loads(json_path.read_text())
    combined_jobs.extend(job_list)

if missing_files:
    print(f"[WARNING] {len(missing_files)} pair(s) had no JSON file on disk and were "
          f"skipped: {missing_files}")
    print("Re-run Part A's generation step for the shortlist before combining if these "
          "are needed.")

# Sanity check: job names must be unique, or the server may not distinguish
# drafts correctly once imported
names = [j["name"] for j in combined_jobs]
if len(names) != len(set(names)):
    dupes = [n for n in set(names) if names.count(n) > 1]
    print(f"[WARNING] Duplicate job names detected, this should not happen given "
          f"pair_id-based naming: {dupes}")

combined_path = Path("/content/ms3_step4/MS3_step4_shortlist_35_combined.json")
combined_path.write_text(json.dumps(combined_jobs, indent=2))

n_included = len(shortlist) - len(missing_files)
print(f"\nCombined {n_included} jobs into one file -> {combined_path}")
print(f"File size: {combined_path.stat().st_size / 1024:.1f} KB")
print(f"Well within AF3 server's documented 100-jobs-per-file limit.")

try:
    from google.colab import files
    files.download(str(combined_path))
except ImportError:
    print(f"Not in Colab -- file saved locally at: {combined_path}")


Combined 35 jobs into one file -> /content/ms3_step4/MS3_step4_shortlist_35_combined.json
File size: 30.2 KB
Well within AF3 server's documented 100-jobs-per-file limit.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import json
from pathlib import Path

TARGET_PAIR_IDS = ["P693", "P567", "P557", "P508", "P445", "P432", "P407",
                    "P340", "P178", "P166", "P163", "P146", "P008", "P001"]

FASTA_DIR = Path("/content/ms3_step4/af3_inputs")

combined_jobs = []
missing = []
for pair_id in TARGET_PAIR_IDS:
    matches = list(FASTA_DIR.glob(f"{pair_id}_af3input.json"))
    if not matches:
        missing.append(pair_id)
        continue
    job_list = json.loads(matches[0].read_text())
    combined_jobs.extend(job_list)

if missing:
    print(f"[WARNING] No local input JSON found for: {missing}")
    print("These pairs may not have been generated yet, or FASTA_DIR doesn't match "
          "where Part A actually wrote them -- check the path above if this list "
          "isn't empty.")

names = [j["name"] for j in combined_jobs]
if len(names) != len(set(names)):
    dupes = [n for n in set(names) if names.count(n) > 1]
    print(f"[WARNING] Duplicate job names in the combined set: {dupes}")

combined_path = Path("/content/ms3_step4/MS3_step4_14pairs_combined.json")
combined_path.write_text(json.dumps(combined_jobs, indent=2))
print(f"\nCombined {len(combined_jobs)}/{len(TARGET_PAIR_IDS)} jobs -> {combined_path}")
print(f"File size: {combined_path.stat().st_size / 1024:.1f} KB")

try:
    from google.colab import files
    files.download(str(combined_path))
except ImportError:
    print(f"Not in Colab -- file saved locally at: {combined_path}")


Combined 14/14 jobs -> /content/ms3_step4/MS3_step4_14pairs_combined.json
File size: 11.7 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## [MANUAL STEP, REPEAT DAILY] Submit Today's Batch to AlphaFold3 Server

For each pair listed above:

1. Go to **https://alphafoldserver.com**, upload the corresponding JSON file
2. Submit and wait for completion
3. Download the result `.zip`, unzip into the matching folder under `af3_results/<pair_id>/`
4. Once you've submitted today's batch, mark them below so tomorrow's batch
   picks up where you left off

Marked 15 pairs as submitted on 2026-08-16
Come back after downloading results and run Part C to ingest them,
then re-run Part B tomorrow for the next batch.


## Part C — Ingest Available Results

Checks `af3_results/` for any completed structures and updates the manifest.
Safe to re-run any time — already-ingested pairs are skipped.

In [28]:
import re
import shutil
from pathlib import Path
from typing import Optional

_required = ["BASE_DIR", "RESULTS_DIR", "MANIFEST_PATH", "check_and_ingest_results", "pd", "DRIVE_ROOT"]
_missing = [name for name in _required if name not in dir()]
if _missing:
    raise RuntimeError(f"Missing: {_missing}. Run Part 0's Drive-mount cell first.")

if DRIVE_ROOT is None:
    raise SystemExit("[BLOCKED] Drive isn't connected -- re-run Part 0's mount cell.")

MY_DRIVE_ROOT = Path("/content/drive/My Drive")
# AF3 lowercases job names when creating output folders (confirmed directly:
# ms3_step4_p001, not P001) -- match case-insensitively and normalize the
# extracted ID back to the manifest's uppercase P### convention
PAIR_ID_PATTERN = re.compile(r"(p\d{3})", re.IGNORECASE)

def extract_pair_id(name: str) -> Optional[str]:
    match = PAIR_ID_PATTERN.search(name)
    return match.group(1).upper() if match else None

source_dir = MY_DRIVE_ROOT / "AF3_Results" / "Step4AF3Results"
if not source_dir.exists():
    print(f"[BLOCKED] {source_dir} not found.")
else:
    print(f"Source directory: {source_dir}")
    subfolders = sorted(p for p in source_dir.iterdir() if p.is_dir())
    print(f"Contains {len(subfolders)} subfolder(s)")

    synced = 0
    unrecognized = []
    incomplete = []

    for folder in subfolders:
        pair_id = extract_pair_id(folder.name)
        if pair_id is None:
            unrecognized.append(folder.name)
            continue

        has_cif = bool(list(folder.glob("*model_0.cif")))
        if not has_cif:
            incomplete.append(folder.name)
            continue

        dest = RESULTS_DIR / pair_id
        if dest.exists() and list(dest.glob("*model_0.cif")):
            continue
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(folder, dest)
        synced += 1

    print(f"\nSynced {synced} pair(s) into {RESULTS_DIR}")
    if unrecognized:
        print(f"[WARNING] {len(unrecognized)} folder(s) had no recognizable pair ID: {unrecognized}")
    if incomplete:
        print(f"[WARNING] {len(incomplete)} folder(s) missing model_0.cif (incomplete/truncated): {incomplete}")

manifest_df = pd.read_csv(MANIFEST_PATH)
manifest_df = check_and_ingest_results(manifest_df)
manifest_df.to_csv(MANIFEST_PATH, index=False)

print("\nStatus across all pairs:")
print(manifest_df["status"].value_counts().to_string())

n_ready = (manifest_df["status"] == "structure_ready").sum()
n_total = len(manifest_df[manifest_df["status"] != "skipped_no_sequence"])
print(f"\n{n_ready}/{n_total} pairs have a usable structure.")

Source directory: /content/drive/My Drive/AF3_Results/Step4AF3Results
Contains 35 subfolder(s)

Synced 35 pair(s) into /content/ms3_step4/af3_results
2 pair(s) newly confirmed with a downloaded structure

Status across all pairs:
status
pending            917
submitted           13
structure_ready      2

2/932 pairs have a usable structure.


In [ ]:
def check_and_ingest_results(manifest_df: pd.DataFrame) -> pd.DataFrame:
    """
    Checks RESULTS_DIR directly for a real model_0.cif per pair, rather than
    only checking pairs whose manifest status already says "submitted".
    Status tracking can drift out of sync with what's actually on disk (e.g.
    if a pair was submitted and synced outside the notebook's own tracked
    flow), so this treats the filesystem as ground truth and updates any
    pending/submitted row that has a real structure available, regardless
    of what its current status column says.
    """
    updated = manifest_df.copy()
    n_newly_ready = 0
    for idx, row in updated.iterrows():
        if row["status"] == "structure_ready":
            continue  # already correctly marked, nothing to do
        if row["status"] == "skipped_no_sequence":
            continue  # never had a valid input, not eligible

        folder = Path(row["expected_result_folder"]) if row["expected_result_folder"] else None
        if folder and folder.exists():
            cifs = list(folder.glob("*model_0.cif"))
            if cifs:
                updated.at[idx, "downloaded"] = True
                updated.at[idx, "status"] = "structure_ready"
                n_newly_ready += 1

    print(f"{n_newly_ready} pair(s) newly confirmed with a downloaded structure")
    return updated

manifest_df = pd.read_csv(MANIFEST_PATH)
manifest_df = check_and_ingest_results(manifest_df)
manifest_df.to_csv(MANIFEST_PATH, index=False)

print("\nStatus across all pairs:")
print(manifest_df["status"].value_counts().to_string())

n_ready = (manifest_df["status"] == "structure_ready").sum()
n_total = len(manifest_df[manifest_df["status"] != "skipped_no_sequence"])
print(f"\n{n_ready}/{n_total} pairs have a usable structure.")
if n_ready > 0:
    ready_ids = manifest_df[manifest_df["status"] == "structure_ready"]["pair_id"].tolist()
    print(f"Ready: {ready_ids}")

## Part D — Export Final Structure Manifest

Once all pairs show `structure_ready`, export the manifest for Step 5
(groove geometry) and Step 6 (PRODIGY affinity).

In [ ]:
FINAL_COLUMNS = ["pair_id", "allele", "locus", "peptide", "epitope_id",
                  "antigen", "donors", "ic50", "expected_result_folder", "status"]
final_manifest = manifest_df.reindex(columns=FINAL_COLUMNS)

out_path = BASE_DIR / "step4_final_manifest.csv"
final_manifest.to_csv(out_path, index=False)

n_ready = (manifest_df["status"] == "structure_ready").sum()
n_total = len(manifest_df[manifest_df["status"] != "skipped_no_sequence"])
print(f"Step 4 status: {n_ready}/{n_total} structures ready")
print(f"Manifest saved: {out_path}")

try:
    from google.colab import files
    files.download(str(out_path))
except ImportError:
    pass

if n_ready < n_total:
    print(f"\n{n_total - n_ready} pair(s) still pending — repeat Parts B-C until all are ready.")
else:
    print("\nAll structures ready. Proceed to Step 5 (groove geometry) and Step 6 (PRODIGY affinity).")
final_manifest.head(10)